In [11]:
import pandas as pd
df =pd.read_csv("../data/milestone1_ayesha-naaz.csv")
df.head()

,id,sender,subject,body,priority,triage_label,clean_text,triage
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,reminder the client meeting is scheduled at t...,respond_or_act
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,your invoice of inr is due on please pay to ...,respond_or_act
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,reminder the client meeting is scheduled at t...,respond_or_act
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,hello team please find the attached weekly rep...,respond_or_act
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,hello team please find the attached weekly rep...,respond_or_act


In [12]:
df.shape


(200, 8)

In [8]:
# import pandas as pd
# df =pd.read_csv("../data/sample_emails_with_triage_200.csv")
# df.head()


import os

print(os.path.abspath("../data/sample_emails_with_triage_200.csv"))


c:\Users\skgha\OneDrive\Desktop\Email Assisstant\infosys-langgraph-email-assistant-group2\data\sample_emails_with_triage_200.csv


In [13]:
def triage_rule(text):
    t = str(text)

    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return 'notify_human'

    if any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return 'ignore'

    if any(k in t for k in ['invoice','payment','overdue','due on','meeting']):
        return 'respond_or_act'

    return 'respond_or_act'


In [14]:
df['triage'] = df['clean_text'].apply(triage_rule)
df[['clean_text','triage']].head()

#The triage function is applied to the cleaned text

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,respond_or_act
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


In [7]:
df.columns


Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label'], dtype='object')

In [15]:
eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label,clean_text,triage
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore,your order has been shipped and is expected t...,respond_or_act
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human,hi dont miss our sale with discounts up to on...,ignore
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human,notice your account will be locked unless veri...,respond_or_act
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond,reminder the client meeting is scheduled at t...,respond_or_act
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond,your order has been shipped and is expected t...,respond_or_act


In [16]:
#create a column of ideal_response(if elif logic)

def ideal_response(text):
    """
    Returns the ideal response based on email content.
    """
    t = str(text).lower()  # ensure lowercase for matching

    # Security-related emails
    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return "Please escalate this security issue to the IT team immediately."

    # Marketing / promotion emails
    elif any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return "No action needed. This is a promotional email."

    # Payment / invoice / meeting emails
    elif any(k in t for k in ['invoice', 'payment', 'overdue', 'due on', 'meeting']):
        return "Please respond appropriately to the client regarding payment or schedule."

    # Default response
    else:
        return "Please read the email and respond as necessary."


In [17]:
df['ideal_response'] = df['clean_text'].apply(ideal_response)
df[['clean_text', 'triage', 'ideal_response']].head()


,clean_text,triage,ideal_response
0,reminder the client meeting is scheduled at t...,respond_or_act,Please respond appropriately to the client reg...
1,your invoice of inr is due on please pay to ...,respond_or_act,Please respond appropriately to the client reg...
2,reminder the client meeting is scheduled at t...,respond_or_act,Please respond appropriately to the client reg...
3,hello team please find the attached weekly rep...,respond_or_act,Please read the email and respond as necessary.
4,hello team please find the attached weekly rep...,respond_or_act,Please read the email and respond as necessary.


In [ ]:
Create a new CSV file, naming it as evaluation for scoring emails with minimum 100 mails. 
Load the file and run your existing agent from milestone1 and generate outputs and save in another CSV file, and create a simple rule-based scoring like tone score, accent score, clarity score.

In [1]:
import pandas as pd
import re


In [4]:
sample_data = pd.read_csv("../data/email_evaluation_dataset_ayesha.csv")
sample_data.head()


,email_text,expected_action,expected_tone
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,urgent
2,We are pleased to inform you that your interns...,respond,polite
3,Happy New Year! Wishing you success and good h...,ignore,neutral
4,Please find attached the minutes of yesterday’...,ignore,neutral


In [5]:
sample_data['clean_text'] = (
    sample_data['email_text']
    .astype(str)
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

sample_data[['email_text', 'clean_text']].head()


,email_text,clean_text
0,Reminder: Client meeting scheduled for Jan 10 ...,reminder client meeting scheduled for jan at ...
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,your invoice inv is due on jan kindly make th...
2,We are pleased to inform you that your interns...,we are pleased to inform you that your interns...
3,Happy New Year! Wishing you success and good h...,happy new year wishing you success and good he...
4,Please find attached the minutes of yesterday’...,please find attached the minutes of yesterdays...


In [6]:
def triage_rule(text):
    t = str(text)

    if any(k in t for k in ['password', 'reset', 'security', 'login']):
        return 'notify_human'

    if any(k in t for k in ['promotion', 'sale', 'offer', 'newsletter']):
        return 'ignore'

    if any(k in t for k in ['invoice', 'payment', 'meeting', 'due', 'submit']):
        return 'respond_or_act'

    return 'respond_or_act'


In [7]:
sample_data['predicted_action'] = sample_data['clean_text'].apply(triage_rule)

sample_data[['email_text', 'expected_action', 'predicted_action']].head()


,email_text,expected_action,predicted_action
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,respond_or_act
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,respond_or_act
2,We are pleased to inform you that your interns...,respond,respond_or_act
3,Happy New Year! Wishing you success and good h...,ignore,respond_or_act
4,Please find attached the minutes of yesterday’...,ignore,respond_or_act


In [8]:
def tone_score(text):
    if any(w in text for w in ['urgent', 'immediately', 'asap']):
        return 5
    if any(w in text for w in ['please', 'kindly', 'thank']):
        return 4
    return 3


In [9]:
import re

def accent_score(text):
    if re.search(r'[^a-zA-Z0-9 .,]', text):
        return 3
    return 5


In [10]:
def clarity_score(text):
    words = len(text.split())
    if words < 5:
        return 2
    if words > 40:
        return 3
    return 5


In [11]:
sample_data['tone_score'] = sample_data['clean_text'].apply(tone_score)
sample_data['accent_score'] = sample_data['clean_text'].apply(accent_score)
sample_data['clarity_score'] = sample_data['clean_text'].apply(clarity_score)

sample_data.head()


,email_text,expected_action,expected_tone,clean_text,predicted_action,tone_score,accent_score,clarity_score
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite,reminder client meeting scheduled for jan at ...,respond_or_act,4,5,5
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,urgent,your invoice inv is due on jan kindly make th...,respond_or_act,4,5,5
2,We are pleased to inform you that your interns...,respond,polite,we are pleased to inform you that your interns...,respond_or_act,4,5,5
3,Happy New Year! Wishing you success and good h...,ignore,neutral,happy new year wishing you success and good he...,respond_or_act,3,5,5
4,Please find attached the minutes of yesterday’...,ignore,neutral,please find attached the minutes of yesterdays...,respond_or_act,4,5,5
